# Tải model dịch Hán → Việt để chạy offline trên L4 x4

Chạy notebook này khi **Internet On**, **không cần GPU** (Accelerator: None).

- Tải model về `/kaggle/working/<tên-model>/` kèm manifest SHA-256 để kiểm tra khi chạy offline.
- Sau khi chạy xong: **Save Version** → Add Output này làm Input cho notebook `2_dich_han_viet.ipynb`.
- Mặc định **Qwen/Qwen2.5-7B-Instruct** (~15,2 GB BF16): vừa 1 GPU L4 24 GB; notebook offline chạy 4 replica, mỗi GPU một replica.
- Đổi `MODEL_ID` nếu muốn model khác. Yêu cầu: model **chat/instruct** (có chat template), định dạng safetensors hoặc `.bin`.
- Model > ~18 GB có thể vượt giới hạn Output của Kaggle; kiểm tra dung lượng in ra ở cuối.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"], check=True)

import os
# Model cần đăng nhập (Llama, Gemma...): Add-ons -> Secrets -> HF_TOKEN.
# Token chỉ đọc từ secret, không in ra log.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN").strip()
    if token:
        os.environ["HF_TOKEN"] = token
    del token
except Exception:
    print("Không có secret HF_TOKEN; dùng token môi trường nếu có.")


In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
import hashlib, json, re, shutil

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
REVISION = "main"  # có thể ghim commit/tag để tái lập chính xác

base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
folder = re.sub(r"[^A-Za-z0-9._-]+", "-", MODEL_ID.split("/")[-1])
destination = base / folder

info = HfApi().model_info(MODEL_ID, revision=REVISION)
print("Model:", MODEL_ID)
print("Revision:", info.sha)
print("Đích:", destination)

snapshot_download(
    MODEL_ID,
    revision=info.sha,
    local_dir=str(destination),
    ignore_patterns=["*.gguf", "*.pth", "*.onnx", "*.msgpack", "original/*"],
)
assert (destination / "config.json").is_file(), "Thiếu config.json"
assert (destination / "tokenizer_config.json").is_file(), "Thiếu tokenizer_config.json"
assert (destination / "tokenizer.json").is_file() or (destination / "tokenizer.model").is_file(), \
    "Thiếu tokenizer.json/tokenizer.model"
assert list(destination.glob("*.safetensors")) or list(destination.glob("*.bin")), "Thiếu trọng số model"

index = destination / "model.safetensors.index.json"
if index.exists():
    for shard in set(json.loads(index.read_text())["weight_map"].values()):
        assert (destination / shard).is_file(), f"Thiếu shard: {shard}"

# Một số phiên bản huggingface_hub tạo symlink vào cache; copy thành file thật
# để Output Add Input không bị hỏng.
links = [p for p in destination.rglob("*") if p.is_symlink()]
for path in links:
    target = path.resolve()
    if path.is_file() or target.is_file():
        path.unlink()
        shutil.copy2(target, path)
assert not any(p.is_symlink() for p in destination.rglob("*")), "Vẫn còn symlink trong thư mục model"
shutil.rmtree(destination / ".cache", ignore_errors=True)

files = []
for path in sorted(destination.rglob("*")):
    if not path.is_file() or ".cache" in path.parts or path.name == "model-manifest.json":
        continue
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    files.append(dict(file=str(path.relative_to(destination)), bytes=path.stat().st_size,
                      sha256=digest.hexdigest()))

manifest = dict(model_id=MODEL_ID, revision=info.sha, files=files)
(destination / "model-manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\nOutput:", destination)
print("Số file:", len(files), "| Dung lượng GiB:", round(sum(f["bytes"] for f in files) / 1024**3, 2))
print("Đã ghi model-manifest.json")
print("\nBước tiếp theo: Save Version để lưu Output, rồi Add Input vào notebook 2_dich_han_viet.ipynb.")


In [ ]:
# Kiểm tra nhanh tokenizer/chat template ngay tại notebook tải model (không cần GPU).
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(str(destination), local_files_only=True)
probe = tok.apply_chat_template(
    [{"role": "system", "content": "Bạn là dịch giả."},
     {"role": "user", "content": "Dịch sang tiếng Việt."}],
    tokenize=False, add_generation_prompt=True,
)
assert probe.strip(), "Chat template rỗng"
print("Tokenizer OK:", type(tok).__name__, "| Vocab:", len(tok))
print("Chat template (200 ký tự đầu):", probe[:200].replace("\n", "\\n"))
